In [1]:
import os
import sys
import time
import json
import logging
import subprocess
import pythoncom
import win32com.client
from pathlib import Path
from tqdm import tqdm


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SOURCE_DIR  = r"E:\Callproject\assembled"
OUTPUT_DIR  = r"E:\Callproject\extracted_json"
#SCRIPT_TEMPLATE = Path(__file__).parent / "extract_text.jsx"

TIMEOUT_PER_FILE = 120
MAX_CONSECUTIVE_ERRORS = 100  # Lowered to trigger "Hard Reset" sooner
#PROGRESS_LOG = Path(__file__).parent / "batch_progress.log"

# If the JSX file is in the same folder as your notebook:
SCRIPT_TEMPLATE = Path.cwd() / "extract_text.jsx"
PROGRESS_LOG = Path.cwd() / "batch_progress.log"

CACHE_FILE = Path.cwd() / "file_list_cache.txt"


In [3]:
# ---------------------------------------------------------------------------
# Logging & Process Management
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        #logging.FileHandler(Path(__file__).parent / "batch_extract.log", encoding="utf-8"),
        logging.FileHandler(Path.cwd() / "batch_extract.log", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger(__name__)

def kill_indesign():
    """Forcefully terminates InDesign to clear hung modal dialogs or zombie processes."""
    log.warning("Triggering hard reset: Killing InDesign.exe process...")
    try:
        subprocess.run(["taskkill", "/F", "/IM", "InDesign.exe"], capture_output=True)
        time.sleep(5) # Allow OS to release file handles
    except Exception as e:
        log.error(f"Failed to kill InDesign: {e}")


In [4]:
# ---------------------------------------------------------------------------
# COM / InDesign Helpers
# ---------------------------------------------------------------------------

def get_indesign():
    """Connect to or launch InDesign with improved version-independent handling."""
    progids = ["InDesign.Application.2026", "InDesign.Application.2025", "InDesign.Application"]
    
    # Try connecting to active instance
    for pid in progids:
        try:
            app = win32com.client.GetActiveObject(pid)
            log.info(f"Connected to {pid}")
            return app
        except: continue

    # Launch new instance if none found
    for pid in progids:
        try:
            app = win32com.client.Dispatch(pid)
            log.info(f"Launched new instance: {pid}")
            time.sleep(8) # Longer wait for legacy environment setup
            return app
        except: continue

    raise RuntimeError("InDesign could not be started. Check installation.")

def run_script(app, script: str) -> tuple:
    """Execute JSX via COM. 1246973031 = JavaScript/ExtendScript."""
    try:
        app.DoScript(script, 1246973031)
        return True, "OK"
    except Exception as e:
        return False, str(e)


In [5]:
# ---------------------------------------------------------------------------
# Progress & File Discovery
# ---------------------------------------------------------------------------

def load_progress():
    return set(line.strip() for line in PROGRESS_LOG.open("r", encoding="utf-8") if line.strip()) if PROGRESS_LOG.exists() else set()

def record_progress(path: str):
    with PROGRESS_LOG.open("a", encoding="utf-8") as f: f.write(path + "\n")

def build_script(template: str, indd_path: Path, json_path: Path) -> str:
    """
    Prepares the JSX script by injecting paths formatted for ExtendScript.
    Using .as_posix() converts 'E:\\Data' to 'E:/Data', which is safer for JSX.
    """
    indd_uri = indd_path.absolute().as_posix()
    json_uri = json_path.absolute().as_posix()
    
    header = (
        f'var inddPath = "{indd_uri}";\n'
        f'var jsonPath = "{json_uri}";\n\n'
    )
    return header + template


In [6]:

# ---------------------------------------------------------------------------
# Main Logic
# ---------------------------------------------------------------------------

def main():

    if CACHE_FILE.exists():
        log.info(f"Loading file list from cache: {CACHE_FILE}")
        with CACHE_FILE.open("r", encoding="utf-8") as f:
            # Filter out any empty lines or whitespace
            all_files = [Path(line.strip()) for line in f if line.strip()]
    else:
        # 2. If no cache, perform the slow scan
        log.info(f"No cache found. Scanning {SOURCE_DIR} (this may take a few minutes)...")
        all_files = sorted([
            f for f in Path(SOURCE_DIR).rglob("*.indd") 
            if not f.name.startswith("._")                     # Ignore Mac metadata
            and "(broken)" not in str(f)                        # Skip known-bad folders
        ])
        
        # 3. Save the results to the cache file
        log.info(f"Scan complete. Saving {len(all_files)} paths to cache.")
        with CACHE_FILE.open("w", encoding="utf-8") as f:
            for path in all_files:
                f.write(str(path) + "\n")

    # 4. Filter out files already processed (using your batch_progress.log)
    done = load_progress()
    pending = [f for f in all_files if str(f) not in done]
    
    log.info(f"Total Files: {len(all_files)} | Remaining to Process: {len(pending)}")

    log.info("Starting InDesign Batch Extraction (Page-Based Mode)")
    jsx_template = SCRIPT_TEMPLATE.read_text(encoding="utf-8")
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    done = load_progress()
    pending = [f for f in all_files if str(f) not in done]
    
    log.info(f"Total: {len(all_files)} | To Process: {len(pending)}")

    pythoncom.CoInitialize()
    try:
        app = get_indesign()
    except Exception as e:
        log.error(e)
        return

    consecutive_errors = 0
    
    with tqdm(total=len(pending), unit="file", desc="Extracting") as bar:
        for indd_path in pending:
            # Derive output path (e.g., E:\...\file.indd -> E:\...\file.json)
            rel = indd_path.relative_to(SOURCE_DIR)
            json_path = Path(OUTPUT_DIR) / rel.with_suffix(".json")
            json_path.parent.mkdir(parents=True, exist_ok=True)

            # 1. Run the script
            full_script = build_script(jsx_template, indd_path, json_path)
            ok, msg = run_script(app, full_script)

            # 2. Validation: Verify JSON and structural integrity
            if ok:
                if not json_path.exists():
                    ok, msg = False, "Output file missing"
                else:
                    try:
                        with json_path.open("r", encoding="utf-8") as f:
                            data = json.load(f)
                            if "pages" not in data:
                                ok, msg = False, "JSON missing 'pages' key"
                    except Exception as e:
                        ok, msg = False, f"JSON Corrupt: {e}"

            # 3. Handling the Outcome (OUTSIDE the validation logic)
            if ok:
                record_progress(str(indd_path))
                consecutive_errors = 0
            else:
                consecutive_errors += 1
                
                # Check the debug log or the returned message for specific legacy issues
                if "InDesign Open Error" in msg or "Cannot open the file" in msg:
                    log.warning(f"SKIPPED (Legacy/Corrupt): {indd_path.name}")
                else:
                    log.warning(f"FAIL: {indd_path.name} - {msg}")
                
                # Reset InDesign only if it's truly stuck (e.g., every 20 consecutive fails)
                if consecutive_errors > 0 and consecutive_errors % 20 == 0:
                    kill_indesign()
                    pythoncom.CoInitialize()
                    app = get_indesign()

            bar.update(1)
            
            # 4. Global safety break
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                log.error(f"Aborting: {consecutive_errors} consecutive failures.")
                break

    pythoncom.CoUninitialize()
    log.info("Process complete.")


In [7]:
if __name__ == "__main__":
    main()

2026-04-02 22:55:30,273  INFO      Loading file list from cache: D:\development\2026_python\2026_thecall\cdwork\extract_indd_gemini\file_list_cache.txt
2026-04-02 22:55:30,678  INFO      Total Files: 74249 | Remaining to Process: 72614
2026-04-02 22:55:30,679  INFO      Starting InDesign Batch Extraction (Page-Based Mode)
2026-04-02 22:55:30,695  INFO      Total: 74249 | To Process: 72614
2026-04-02 22:55:37,559  INFO      Launched new instance: InDesign.Application.2026


Extracting:   0%|                                                                          | 0/72614 [00:00<?, ?file/s]

2026-04-02 22:55:45,831  WARNING   FAIL: Long, Kiana L.indd - Output file missing


Extracting:   3%|█▋                                                           | 1942/72614 [11:29<7:07:24,  2.76file/s]

2026-04-02 23:07:15,467  WARNING   FAIL: Dunson Carrie.indd - Output file missing


...

2026-04-03 02:36:29,563  INFO      Launched new instance: InDesign.Application.2026


Extracting:  41%|███████████████████████▏                                 | 29571/72614 [3:40:51<37:02:51,  3.10s/file]

2026-04-03 02:36:37,567  ERROR     Aborting: 100 consecutive failures.


Extracting:  41%|███████████████████████▌                                  | 29571/72614 [3:40:52<5:21:29,  2.23file/s]

2026-04-03 02:36:37,580  INFO      Process complete.


In [8]:
if __name__ == "__main__":
    main()

2026-04-03 02:56:32,731  INFO      Loading file list from cache: D:\development\2026_python\2026_thecall\cdwork\extract_indd_gemini\file_list_cache.txt
2026-04-03 02:56:33,248  INFO      Total Files: 74249 | Remaining to Process: 43256
2026-04-03 02:56:33,249  INFO      Starting InDesign Batch Extraction (Page-Based Mode)
2026-04-03 02:56:33,370  INFO      Total: 74249 | To Process: 43256
2026-04-03 02:56:33,379  INFO      Launched new instance: InDesign.Application.2026


Extracting:   0%|                                                                          | 0/43256 [00:00<?, ?file/s]

2026-04-03 02:56:41,488  WARNING   FAIL: Long, Kiana L.indd - Output file missing


Extracting:   0%|                                                                | 1/43256 [00:00<1:16:45,  9.39file/s]

2026-04-03 02:56:41,580  WARNING   FAIL: Dunson Carrie.indd - Output file missing
2026-04-03 02:56:41,658  WARNING   FAIL: Dunson Carrie.indd - Output file missing


...

2026-04-03 02:58:27,619  WARNING   FAIL: _-religious log.indd - Output file missing
2026-04-03 02:58:27,678  WARNING   FAIL: _-religious log.indd - Output file missing
2026-04-03 02:58:27,679  WARNING   Triggering hard reset: Killing InDesign.exe process...
2026-04-03 02:58:43,822  INFO      Launched new instance: InDesign.Application.2026


Extracting:   0%|▏                                                            | 100/43256 [02:10<42:34:08,  3.55s/file]

2026-04-03 02:58:51,826  ERROR     Aborting: 100 consecutive failures.


Extracting:   0%|▏                                                            | 100/43256 [02:10<15:38:15,  1.30s/file]

2026-04-03 02:58:51,831  INFO      Process complete.


In [9]:
if __name__ == "__main__":
    main()
    

2026-04-03 03:08:19,490  INFO      Loading file list from cache: D:\development\2026_python\2026_thecall\cdwork\extract_indd_gemini\file_list_cache.txt
2026-04-03 03:08:19,903  INFO      Total Files: 74249 | Remaining to Process: 43256
2026-04-03 03:08:19,904  INFO      Starting InDesign Batch Extraction (Page-Based Mode)
2026-04-03 03:08:19,943  INFO      Total: 74249 | To Process: 43256
2026-04-03 03:08:19,950  INFO      Launched new instance: InDesign.Application.2026


Extracting:   0%|                                                                          | 0/43256 [00:00<?, ?file/s]

2026-04-03 03:08:28,036  WARNING   FAIL: Long, Kiana L.indd - Output file missing
2026-04-03 03:08:28,065  WARNING   FAIL: Dunson Carrie.indd - Output file missing


Extracting:   0%|                                                                  | 2/43256 [00:00<40:32, 17.78file/s]

2026-04-03 03:08:28,114  WARNING   FAIL: Dunson Carrie.indd - Output file missing
2026-04-03 03:08:28,167  WARNING   FAIL: Dunson Carrie.indd - Output file missing


...

2026-04-03 03:10:10,287  WARNING   FAIL: _-religious log.indd - Output file missing
2026-04-03 03:10:10,288  WARNING   Triggering hard reset: Killing InDesign.exe process...
2026-04-03 03:10:26,514  INFO      Launched new instance: InDesign.Application.2026


Extracting:   0%|▏                                                            | 100/43256 [02:06<37:11:44,  3.10s/file]

2026-04-03 03:10:34,519  ERROR     Aborting: 100 consecutive failures.


Extracting:   0%|▏                                                            | 100/43256 [02:06<15:10:21,  1.27s/file]

2026-04-03 03:10:34,524  INFO      Process complete.
